# 01b — Preprocess: Nexdata In-Cabin Passenger Human Body Detection

Downloads and preprocesses [3360 Images of 168 People - Passenger Behavior Human Body Detection Data](https://huggingface.co/datasets/Nexdata/3360-Images-of-168-People-Passenger-Behavior-Human-Body-Detection-Data).

In-cabin camera images with rectangular bounding box annotations (JSON format) around human bodies.

| | |
|---|---|
| **Source** | HuggingFace (Nexdata) |
| **Size** | ~3,360 images (168 people x 20 images) |
| **Classes** | `person` |
| **Format** | JSON rectangle annotations -> YOLO |
| **Scenes** | In-cabin cameras (day SUV, evening SUV, night MPV) |

In [ ]:
!pip install albumentations huggingface_hub -q

import os, shutil, glob, random, json
import numpy as np
import matplotlib.pyplot as plt
import cv2
from collections import Counter

print('Setup done!')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/AI_TRAINING/GreenVision'
OUTPUT_DIR = '/content/dataset_nexdata'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')

## 1. Download from HuggingFace

In [ ]:
from huggingface_hub import snapshot_download

RAW_DIR = '/content/raw_nexdata'

if not os.path.exists(RAW_DIR):
    print('Downloading from HuggingFace...')
    snapshot_download(
        repo_id='Nexdata/3360-Images-of-168-People-Passenger-Behavior-Human-Body-Detection-Data',
        repo_type='dataset',
        local_dir=RAW_DIR,
    )
    print('Done!')
else:
    print(f'Already downloaded: {RAW_DIR}')

## 2. Inspect Structure

In [ ]:
# The structure is:
# APY240618010/
#   day_SUV/ID0489_China_female_33/01/Calling_bright/000090.jpg + .json
#   evening_SUV/...
#   night_MPV/...

# Each JSON has format:
# {"dataList": [{"id":1, "shapeType":"rectangle", "label":"Human[&]body",
#                "coordinates": [[x1,y1],[x2,y2]]}]}

# Count files
all_jpg = glob.glob(os.path.join(RAW_DIR, '**', '*.jpg'), recursive=True)
all_json = glob.glob(os.path.join(RAW_DIR, '**', '*.json'), recursive=True)

print(f'Images (jpg): {len(all_jpg)}')
print(f'Annotations (json): {len(all_json)}')

# Show sample annotation
if all_json:
    sample = json.load(open(all_json[0]))
    print(f'\nSample annotation ({os.path.basename(all_json[0])}):')
    print(json.dumps(sample, indent=2))

# Show scene breakdown
scenes = set()
for p in all_jpg:
    parts = p.replace(RAW_DIR, '').split(os.sep)
    if len(parts) > 2:
        scenes.add(parts[2])
print(f'\nScenes: {sorted(scenes)}')

## 3. Convert JSON Annotations to YOLO Format

JSON format: `[[x1,y1],[x2,y2]]` (absolute pixel coords) -> YOLO: `class cx cy w h` (normalized).

In [ ]:
# Clean output dir
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(f'{OUTPUT_DIR}/images', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/labels', exist_ok=True)

converted = 0
no_annotation = 0
total_boxes = 0

for jpg_path in all_jpg:
    json_path = jpg_path.replace('.jpg', '.json')

    if not os.path.exists(json_path):
        no_annotation += 1
        continue

    # Read image dimensions
    img = cv2.imread(jpg_path)
    if img is None:
        continue
    h, w = img.shape[:2]

    # Read JSON annotation
    with open(json_path) as f:
        ann = json.load(f)

    yolo_lines = []
    for item in ann.get('dataList', []):
        if item.get('shapeType') != 'rectangle':
            continue
        coords = item.get('coordinates', [])
        if len(coords) < 2:
            continue
        x1, y1 = coords[0]
        x2, y2 = coords[1]

        # Convert to YOLO normalized format
        cx = ((x1 + x2) / 2) / w
        cy = ((y1 + y2) / 2) / h
        bw = (x2 - x1) / w
        bh = (y2 - y1) / h

        # Clip to valid range
        cx = np.clip(cx, 0, 1)
        cy = np.clip(cy, 0, 1)
        bw = np.clip(bw, 0.001, 1)
        bh = np.clip(bh, 0.001, 1)

        yolo_lines.append(f'0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')

    if not yolo_lines:
        no_annotation += 1
        continue

    # Build unique name from path
    rel = jpg_path.replace(RAW_DIR, '').strip(os.sep)
    name = rel.replace(os.sep, '_').replace('.jpg', '')

    # Copy image and write label
    shutil.copy2(jpg_path, f'{OUTPUT_DIR}/images/{name}.jpg')
    with open(f'{OUTPUT_DIR}/labels/{name}.txt', 'w') as f:
        f.write('\n'.join(yolo_lines))

    converted += 1
    total_boxes += len(yolo_lines)

print(f'Converted: {converted} images')
print(f'No annotation: {no_annotation}')
print(f'Total bounding boxes: {total_boxes}')
print(f'Avg boxes/img: {total_boxes/converted:.1f}' if converted > 0 else '')

## 4. Augment (3x)

In [ ]:
import albumentations as A

img_dir = f'{OUTPUT_DIR}/images'
lbl_dir = f'{OUTPUT_DIR}/labels'

imgs = glob.glob(os.path.join(img_dir, '*.jpg'))
print(f'Original images: {len(imgs)}')

aug_flip = A.Compose([
    A.HorizontalFlip(p=1.0),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

aug_bc = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.2, p=1.0),
    A.GaussNoise(var_limit=(10, 30), p=0.7),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

augmented = 0

for img_path in imgs:
    base = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path = os.path.join(lbl_dir, base + '.txt')

    if not os.path.exists(lbl_path):
        continue

    image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

    with open(lbl_path) as f:
        lines = f.read().strip().split('\n')
    bboxes, labels = [], []
    for line in lines:
        parts = line.strip().split()
        if len(parts) == 5:
            labels.append(int(parts[0]))
            bboxes.append([float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])])

    if not bboxes:
        continue

    # Horizontal flip
    try:
        result = aug_flip(image=image, bboxes=bboxes, class_labels=labels)
        cv2.imwrite(os.path.join(img_dir, f'{base}_flip.jpg'),
                    cv2.cvtColor(result['image'], cv2.COLOR_RGB2BGR))
        with open(os.path.join(lbl_dir, f'{base}_flip.txt'), 'w') as f:
            for cls, bb in zip(result['class_labels'], result['bboxes']):
                f.write(f'{cls} {bb[0]:.6f} {bb[1]:.6f} {bb[2]:.6f} {bb[3]:.6f}\n')
        augmented += 1
    except Exception:
        pass

    # Brightness + contrast + noise
    try:
        result = aug_bc(image=image, bboxes=bboxes, class_labels=labels)
        cv2.imwrite(os.path.join(img_dir, f'{base}_bc.jpg'),
                    cv2.cvtColor(result['image'], cv2.COLOR_RGB2BGR))
        with open(os.path.join(lbl_dir, f'{base}_bc.txt'), 'w') as f:
            for cls, bb in zip(result['class_labels'], result['bboxes']):
                f.write(f'{cls} {bb[0]:.6f} {bb[1]:.6f} {bb[2]:.6f} {bb[3]:.6f}\n')
        augmented += 1
    except Exception:
        pass

final_imgs = glob.glob(os.path.join(img_dir, '*.jpg'))
print(f'\nAugmented: +{augmented} images')
print(f'Total: {len(final_imgs)} images ({len(final_imgs)/len(imgs):.1f}x)')

## 5. Stats & Preview

In [ ]:
img_dir = f'{OUTPUT_DIR}/images'
lbl_dir = f'{OUTPUT_DIR}/labels'

imgs = glob.glob(os.path.join(img_dir, '*.jpg'))
lbls = glob.glob(os.path.join(lbl_dir, '*.txt'))

total_boxes = 0
for lbl in lbls:
    with open(lbl) as f:
        total_boxes += len(f.readlines())

print('=' * 45)
print('  Nexdata In-Cabin Dataset Summary')
print('=' * 45)
print(f'  Images:         {len(imgs)}')
print(f'  Labels:         {len(lbls)}')
print(f'  Bounding boxes: {total_boxes}')
print(f'  Avg boxes/img:  {total_boxes/len(lbls):.1f}')
print('=' * 45)

# Sample visualization
CLASS_COLOR = (0, 255, 0)
originals = [p for p in imgs if '_flip' not in os.path.basename(p) and '_bc' not in os.path.basename(p)]
sample = random.sample(originals, min(12, len(originals)))

fig, axes = plt.subplots(3, 4, figsize=(20, 13))
for ax, img_path in zip(axes.flatten(), sample):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    base = os.path.splitext(os.path.basename(img_path))[0]
    lbl = os.path.join(lbl_dir, base + '.txt')
    count = 0
    if os.path.exists(lbl):
        with open(lbl) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cx, cy, bw, bh = map(float, parts[1:5])
                    x1 = int((cx - bw/2) * w)
                    y1 = int((cy - bh/2) * h)
                    x2 = int((cx + bw/2) * w)
                    y2 = int((cy + bh/2) * h)
                    cv2.rectangle(img, (x1, y1), (x2, y2), CLASS_COLOR, 2)
                    count += 1
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'{count} person(s)', fontsize=9)
plt.suptitle('Nexdata — In-Cabin Passenger Detection', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Save to Drive

In [ ]:
drive_output = os.path.join(DRIVE_ROOT, 'datasets', 'nexdata')
if os.path.exists(drive_output):
    shutil.rmtree(drive_output)
shutil.copytree(OUTPUT_DIR, drive_output)

print(f'Saved to: {drive_output}')
print(f'Images: {len(glob.glob(os.path.join(drive_output, "images", "*.jpg")))}')
print(f'Labels: {len(glob.glob(os.path.join(drive_output, "labels", "*.txt")))}')

---
## Done!

Preprocessed dataset saved to:
```
/content/drive/MyDrive/AI_TRAINING/GreenVision/datasets/nexdata/
  images/   — all images (original + augmented)
  labels/   — YOLO format labels (class 0 = person)
```

Ready to merge with other datasets or use for training.